# 03. Florence-2 LoRA 파인튜닝

**목표**
- CORD v2로 Florence-2-base-ft LoRA 파인튜닝
- WandB로 학습 곡선 추적
- 체크포인트 Kaggle output / Google Drive에 저장

**실행 환경**: Kaggle (T4/P100) 또는 Google Colab

In [ ]:
# 레포 클론
!git clone https://github.com/shimtaehun/korean-doc-understanding.git

import sys
sys.path.append("/kaggle/working/korean-doc-understanding")
# Colab이라면:
# sys.path.append("/content/korean-doc-understanding")

In [ ]:
!pip install -q transformers peft accelerate datasets wandb jiwer scikit-learn albumentations pyyaml

In [ ]:
import os
import torch

# WandB API 키 설정
# Kaggle: Add-ons → Secrets → WANDB_API_KEY 추가 후 아래 코드 사용
from kaggle_secrets import UserSecretsClient
os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")

# Colab이라면:
# from google.colab import userdata
# os.environ["WANDB_API_KEY"] = userdata.get('WANDB_API_KEY')

print("torch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음")

## 1. 학습 설정

In [ ]:
import yaml

# 기본 config 로드 후 Kaggle 환경에 맞게 override
with open("/kaggle/working/korean-doc-understanding/configs/train_config.yaml") as f:
    cfg = yaml.safe_load(f)

# Kaggle 환경 맞게 경로 수정
cfg["output"]["checkpoint_dir"] = "/kaggle/working/checkpoints"
cfg["wandb"]["entity"] = "shimtaehun"  # ← WandB 사용자명

# VRAM 부족 시 여기서 조정
cfg["training"]["batch_size"] = 2
cfg["training"]["gradient_accumulation_steps"] = 8  # effective batch = 16
cfg["training"]["epochs"] = 5

print(yaml.dump(cfg, allow_unicode=True))

## 2. 모델 + 데이터 로드

In [ ]:
from datasets import load_dataset
from src.data.dataset import CORDDataset
from src.model.florence_lora import LoRASettings, load_florence_with_lora

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

lora_settings = LoRASettings(
    r=cfg["lora"]["r"],
    alpha=cfg["lora"]["alpha"],
    dropout=cfg["lora"]["dropout"],
    target_modules=cfg["lora"]["target_modules"],
)

model, processor = load_florence_with_lora(
    model_id=cfg["model"]["name"],
    lora_settings=lora_settings,
    device=DEVICE,
)

In [ ]:
hf_dataset = load_dataset(cfg["data"]["hf_dataset"])

from torch.utils.data import DataLoader

train_ds = CORDDataset(hf_dataset["train"], processor, split="train",
                       max_length=cfg["model"]["max_length"], augment=True)
val_ds   = CORDDataset(hf_dataset["validation"], processor, split="validation",
                       max_length=cfg["model"]["max_length"], augment=False)

train_loader = DataLoader(train_ds, batch_size=cfg["training"]["batch_size"],
                          shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=cfg["training"]["batch_size"],
                          shuffle=False, num_workers=2, pin_memory=True)

print(f"train: {len(train_ds)}개 / val: {len(val_ds)}개")

## 3. 학습 실행

In [ ]:
import wandb
from torch.optim import AdamW
from torch.cuda.amp import GradScaler, autocast
from transformers import get_cosine_schedule_with_warmup
from src.training.evaluate import evaluate
from src.training.callbacks import WandBCallback, CheckpointCallback

wandb.init(
    project=cfg["wandb"]["project"],
    entity=cfg["wandb"]["entity"],
    config=cfg,
    name=f"lora-r{cfg['lora']['r']}-lr{cfg['training']['learning_rate']}",
)

optimizer = AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=cfg["training"]["learning_rate"],
    weight_decay=cfg["training"]["weight_decay"],
)
total_steps  = len(train_loader) * cfg["training"]["epochs"]
warmup_steps = int(total_steps * cfg["training"]["warmup_ratio"])
scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)
scaler    = GradScaler(enabled=cfg["training"]["fp16"])

wandb_cb = WandBCallback(log_interval=cfg["wandb"]["log_interval"])
ckpt_cb  = CheckpointCallback(
    checkpoint_dir=cfg["output"]["checkpoint_dir"],
    save_best_only=cfg["output"]["save_best_only"],
)

In [ ]:
accum_steps = cfg["training"]["gradient_accumulation_steps"]

for epoch in range(1, cfg["training"]["epochs"] + 1):
    print(f"\n=== Epoch {epoch}/{cfg['training']['epochs']} ===")
    model.train()
    total_loss = 0.0
    optimizer.zero_grad()

    for step, batch in enumerate(train_loader):
        pixel_values  = batch["pixel_values"].to(DEVICE)
        input_ids     = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels        = batch["labels"].to(DEVICE)

        with autocast(dtype=torch.float16, enabled=cfg["training"]["fp16"]):
            outputs = model(
                pixel_values=pixel_values,
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels,
            )
            loss = outputs.loss / accum_steps

        scaler.scale(loss).backward()

        if (step + 1) % accum_steps == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["training"]["max_grad_norm"])
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()

        total_loss += loss.item() * accum_steps
        wandb_cb.on_step_end(
            loss=total_loss / (step + 1),
            lr=scheduler.get_last_lr()[0],
            epoch=epoch,
        )

    # 검증
    val_metrics = evaluate(model, processor, val_loader, DEVICE)
    epoch_loss  = total_loss / len(train_loader)

    print(f"  loss={epoch_loss:.4f} | field_f1={val_metrics['field_f1']:.4f} | cer={val_metrics['cer']:.4f}")

    wandb_cb.on_epoch_end(epoch, epoch_loss, val_metrics)
    ckpt_cb.on_epoch_end(model, processor, epoch, val_metrics)

wandb_cb.on_train_end(ckpt_cb._best_metric)
print("\n학습 완료!")

## 4. 체크포인트 확인

In [ ]:
import os
ckpt_dir = cfg["output"]["checkpoint_dir"]
for d in sorted(os.listdir(ckpt_dir)):
    size = sum(
        os.path.getsize(os.path.join(ckpt_dir, d, f))
        for f in os.listdir(os.path.join(ckpt_dir, d))
    ) / 1024 / 1024
    print(f"{d}: {size:.1f} MB")